In [ ]:
# import libraries
import pandas as pd
import plotly.graph_objects as go

In [ ]:
# load data
df_prod = pd.read_csv('../data/Global_production_quantity.csv')
df_species = pd.read_csv('../data/CL_FI_SPECIES_GROUPS.csv')
df_countries = pd.read_csv('../data/CL_FI_COUNTRY_GROUPS.csv')

In [ ]:
# preview production data
df_prod.info()

In [ ]:
# preview specie data
df_species.head()

In [ ]:
# preview country data
df_countries.head()

In [ ]:
# filter specie data to keep only seaweed species
mask_columns = ["CPC_Class_Es","CPC_Class_Ar","CPC_Class_Cn","CPC_Class_Ru", "ISSCAAP_Group_Cn", "ISSCAAP_Group_Ru", "CPC_Class_Fr","CPC_Group_Fr","CPC_Group_Es", "CPC_Group_Ar","CPC_Group_Cn","CPC_Group_Ru", "ISSCAAP_Group_Es","ISSCAAP_Group_Ar", "Yearbook_Group_Es", "Yearbook_Group_Ar", "Yearbook_Group_Cn", "Yearbook_Group_Ru", "ISSCAAP_Group_En"]
df_species_filter_alguae = df_species.drop(columns=mask_columns)[df_species["ISSCAAP_Group_Fr"].isin(["Algues rouges", "Algues brunes", "Algues vertes"])]

# get list of seaweed species
list_algae_species = df_species_filter_alguae["3A_Code"].tolist()

# filter production data to keep only seaweed species
df_prod_alguae = df_prod[df_prod["SPECIES.ALPHA_3_CODE"].isin(list_algae_species)]

# rename some columns
df_prod_alguae = df_prod_alguae.rename(columns={"PRODUCTION_SOURCE_DET.CODE": "source_production","COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Année","VALUE" : "Production"})

# rename production origin (as Récolte or Culture)
df_prod_alguae["source_production"] = df_prod_alguae["source_production"].apply(lambda x: "Récolte" if x == "CAPTURE" else "Culture")


In [ ]:
# filter country data to keep relevant columns only
df_countries_filter = df_countries[["UN_Code", 'Name_Fr', "Continent_Group_Fr"]]

# fix French name of South Korea
mask_korea = df_countries_filter["Name_Fr"] == "République de Corée"
df_countries_filter.loc[mask_korea, "Name_Fr"] = "Corée du Sud"

# get list of european countries
df_europe = df_countries_filter[df_countries_filter["Continent_Group_Fr"] == "Europe"]
list_europe = df_europe["UN_Code"].tolist()

In [ ]:
# merge production and country data
df_prod_alguae_country = df_prod_alguae.merge(df_countries_filter, on="UN_Code", how="left")

## Plot European production over years

In [ ]:
# copy data to plot
data_fig2 = df_prod_alguae_country.copy()

# select european countries
data_fig2 = data_fig2[data_fig2["UN_Code"].isin(list_europe)].rename(columns={"COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Année","VALUE" : "Production"})

# set country names or category of countries
mask_uk = data_fig2["Name_Fr"] == "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord"
mask_russia = data_fig2["Name_Fr"] == "Fédération de Russie"
mask_countries = data_fig2["Name_Fr"].isin(["Norvège", "France", "Irlande", "Islande"])
data_fig2["Pays"] = "Autres pays d'Europe"
data_fig2.loc[mask_uk, "Pays"] = "Royaume-Uni"
data_fig2.loc[mask_russia, "Pays"] = "Russie"
data_fig2.loc[mask_countries, "Pays"] = data_fig2.loc[mask_countries, "Name_Fr"]

# group data by year
data_fig2_by_year = data_fig2[["Année", "Pays", "Production"]].groupby(["Année", "Pays"]).sum().reset_index()

### PC version

In [ ]:
# set colors
colors = [('Norvège','#C73175'),('France','#0074E4'),('Irlande','#58B7B0'),('Islande','#D85F9F'), \
    ("Russie",'#F9C06B'), ('Royaume-Uni','#E1D3E9'), ("Autres pays d'Europe",'#F18882')]

# plot production
fig2 = go.Figure()

for name_color_tuple in colors:
    fig2.add_trace(go.Scatter(
        x=data_fig2_by_year[data_fig2_by_year["Pays"]== name_color_tuple[0]]["Année"],
        y=data_fig2_by_year[data_fig2_by_year["Pays"]== name_color_tuple[0]]["Production"],
        mode='lines',
        line=dict(width=0.5, color=name_color_tuple[1]),
        stackgroup='one',
        fillcolor=name_color_tuple[1],
        name=name_color_tuple[0],
        hovertemplate="""Production : %{y}""" + \
        """<extra></extra>""",
        hoverlabel=dict(font=dict(family="Montserrat")),
        )
    )

fig2.update_xaxes(
    showgrid=False,
    ticklabelstandoff=10,
)

fig2.update_yaxes(
    title=dict(
        text="Production (tonnes d'algues)",
        font=dict(size=16)
    ),
    gridcolor="#1D3B6E",
    tickvals=[0, 100000, 200000, 300000, 400000, 500000],
    ticklabelstandoff=20,
)

fig2.update_layout(
    hovermode='x unified',
    font_family="Montserrat",
    font_color="#1D3B6E",
    paper_bgcolor ="#FDF2EE",
    plot_bgcolor ="#FDF2EE",
    legend=dict(
        orientation="h",
        font_family="Montserrat",
        font_color="#113972",
        bgcolor="#FDF2EE",
        entrywidth=150,
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5
    ),
    margin=dict(
        l=20,
        r=20,
        t=30,
        b=20
    ),
    height=400,
    width=900,
)

fig2.show()

In [ ]:
fig2.write_html("../figures/production_europe_total_pc.html", include_plotlyjs="cdn")

### Mobile version

In [ ]:
# set colors
colors = [('Norvège','#C73175'),('France','#0074E4'),('Irlande','#58B7B0'),('Islande','#D85F9F'), \
    ("Russie",'#F9C06B'), ('Royaume-Uni','#E1D3E9'), ("Autres pays d'Europe",'#F18882')]

# plot production
fig2 = go.Figure()

for name_color_tuple in colors:
    fig2.add_trace(go.Scatter(
        x=data_fig2_by_year[data_fig2_by_year["Pays"]== name_color_tuple[0]]["Année"],
        y=data_fig2_by_year[data_fig2_by_year["Pays"]== name_color_tuple[0]]["Production"],
        mode='lines',
        line=dict(width=0.5, color=name_color_tuple[1]),
        stackgroup='one',
        fillcolor=name_color_tuple[1],
        name=name_color_tuple[0],
        hovertemplate="""Production : %{y}""" + \
        """<extra></extra>""",
        hoverlabel=dict(font=dict(family="Montserrat")),
        )
    )

fig2.update_xaxes(
    showgrid=False,
    ticklabelstandoff=10,
)

fig2.update_yaxes(
    title=dict(
        text="Production (tonnes d'algues)",
        font=dict(size=16)
    ),
    gridcolor="#1D3B6E",
    tickvals=[0, 100000, 200000, 300000, 400000, 500000],
    ticklabelstandoff=5,
)

fig2.update_layout(
    hovermode='x unified',
    font_family="Montserrat",
    font_color="#1D3B6E",
    paper_bgcolor ="#FDF2EE",
    plot_bgcolor ="#FDF2EE",
    legend=dict(
        orientation="h",
        font_family="Montserrat",
        font_color="#113972",
        bgcolor="#FDF2EE",
        entrywidth=150,
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5
    ),
    margin=dict(
        l=20,
        r=5,
        t=30,
        b=20
    ),
    height=500,
    width=250,
)

fig2.show()

In [ ]:
fig2.write_html("../figures/production_europe_total_mobile.html", include_plotlyjs="cdn")